# 店舗売上需要予測: 実験 #011 メダル圏突破パック ＆ 2シードアベレージング

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 
  1. ゼロ売上フィルターの誤爆救済（2017年実績ゼロの67系列に限定）
  2. 店舗52（2017-04-20新規開店）のオープン前架空ゼロ売上データの除外
  3. アンバト市設立記念日（8/24）地方祝日フラグ（店舗23, 50）
  4. 2シード・アベレージング（Seed 42 + Seed 2026）によるバリアンス極小化
- **モデル:** 33商品ファミリー別 3大GBDTアンサンブル (LightGBM 50% + CatBoost 30% + XGBoost 20%)
- **ハードウェア最適化:** Intel N100 (4コア/4スレッド CPU 最適化: `n_jobs=4`, `thread_count=4`)
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **結果:** 手元検証 **Val RMSLE `0.39765` (手元大幅改善)**、Public LB `0.39233` (自己ベスト #010: **`0.38968` / 76位** を堅守)


## 2. ライブラリ読み込み & データ準備

In [ ]:
import os
import sys
import glob
import warnings
from datetime import timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning)

def find_data_dir():
    candidates = [
        '/kaggle/input/store-sales-time-series-forecasting',
        '/kaggle/input/competitions/store-sales-time-series-forecasting',
        '003_store-sales/data',
        'data',
        '../data',
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    matches_local = glob.glob('**/train.csv', recursive=True)
    if matches_local:
        return os.path.dirname(matches_local[0])
    raise FileNotFoundError('train.csv が見つかりませんでした。')

data_dir = find_data_dir()
print(f'データ読み込み元: {data_dir}')

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])


## 3. 安全なゼロ系列抽出 & 特徴量構築

In [ ]:
# 安全なゼロ売上系列 (2017年完全ゼロ)
t_2017 = train_df[train_df['date'] >= '2017-01-01']
zero_sum_tst = t_2017.groupby(['store_nbr', 'family'])['sales'].sum()
safe_zero_pairs_tst = set(zero_sum_tst[zero_sum_tst == 0].index)
print(f'テスト用 2017年完全ゼロ系列数: {len(safe_zero_pairs_tst)} / 1,782 ペア')

v_2017 = train_df[(train_df['date'] >= '2017-01-01') & (train_df['date'] < '2017-07-31')]
zero_sum_val = v_2017.groupby(['store_nbr', 'family'])['sales'].sum()
safe_zero_pairs_val = set(zero_sum_val[zero_sum_val == 0].index)

# 全体結合
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])

# カレンダー特徴量
all_df['year'] = all_df['date'].dt.year
all_df['month'] = all_df['date'].dt.month
all_df['day'] = all_df['date'].dt.day
all_df['dayofweek'] = all_df['date'].dt.dayofweek
all_df['is_weekend'] = all_df['dayofweek'].isin([5, 6]).astype(int)
all_df['dayofyear'] = all_df['date'].dt.dayofyear
all_df['weekofyear'] = all_df['date'].dt.isocalendar().week.astype(int)
all_df['is_month_start'] = all_df['date'].dt.is_month_start.astype(int)
all_df['is_month_end'] = all_df['date'].dt.is_month_end.astype(int)

# ドメイン補正: 給与日金曜前倒し (Quincena)
def create_wage_day_features(dates):
    day = dates.dt.day
    is_month_end = dates.dt.is_month_end
    dow = dates.dt.dayofweek
    is_wage = (day == 15) | is_month_end
    is_friday = (dow == 4)
    next_day = dates + pd.Timedelta(days=1)
    next_2day = dates + pd.Timedelta(days=2)
    wage_sat = (next_day.dt.day == 15) | next_day.dt.is_month_end
    wage_sun = (next_2day.dt.day == 15) | next_2day.dt.is_month_end
    is_weekend = dow.isin([5, 6])
    actual_wage_day = (is_wage & ~is_weekend) | (is_friday & (wage_sat | wage_sun))
    actual_wage_day_after = actual_wage_day.shift(1).fillna(False)
    return actual_wage_day.astype(int), actual_wage_day_after.astype(int)

all_df['is_wage_day'], all_df['is_wage_day_after'] = create_wage_day_features(all_df['date'])
all_df['is_ambato_holiday'] = ((all_df['month'] == 8) & (all_df['day'] == 24) & (all_df['store_nbr'].isin([23, 50]))).astype(int)

# 系列ソート
all_df = all_df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
group_promo = all_df.groupby(['store_nbr', 'family'])['onpromotion']

# 特売リード＆ラグ ＆ 特売平常比
all_df['promo_lag_1'] = group_promo.shift(1).fillna(0)
all_df['promo_lag_2'] = group_promo.shift(2).fillna(0)
all_df['promo_lag_7'] = group_promo.shift(7).fillna(0)
all_df['promo_lead_1'] = group_promo.shift(-1).bfill().fillna(0)
all_df['promo_lead_2'] = group_promo.shift(-2).bfill().fillna(0)
all_df['promo_diff_1'] = all_df['onpromotion'] - all_df['promo_lag_1']
all_df['promo_rolling_mean_7'] = group_promo.transform(lambda x: x.rolling(7, min_periods=1).mean())
all_df['promo_rolling_mean_28'] = group_promo.transform(lambda x: x.rolling(28, min_periods=1).mean())
all_df['promo_intensity_7'] = all_df['onpromotion'] / (all_df['promo_rolling_mean_7'] + 1.0)
all_df['promo_intensity_28'] = all_df['onpromotion'] / (all_df['promo_rolling_mean_28'] + 1.0)

# 過去売上ラグ
group_sales = all_df.groupby(['store_nbr', 'family'])['log_sales']
for lag in [1, 2, 3, 7, 14, 21, 28, 364]:
    all_df[f'sales_lag_{lag}'] = group_sales.shift(lag)
all_df['sales_lag_364'] = all_df['sales_lag_364'].fillna(all_df['sales_lag_28'])

lag1 = group_sales.shift(1)
all_df['sales_rolling_mean_7'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(7, min_periods=1).mean())
all_df['sales_rolling_mean_14'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(14, min_periods=1).mean())
all_df['sales_rolling_mean_28'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(28, min_periods=1).mean())
all_df['sales_rolling_std_7'] = lag1.groupby([all_df['store_nbr'], all_df['family']]).transform(lambda x: x.rolling(7, min_periods=1).std()).fillna(0)

features = [
    'store_nbr', 'onpromotion', 'year', 'month', 'day', 'dayofweek', 'is_weekend',
    'dayofyear', 'weekofyear', 'is_month_start', 'is_month_end', 'is_wage_day', 'is_wage_day_after', 'is_ambato_holiday',
    'sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'sales_lag_7', 'sales_lag_14',
    'sales_lag_21', 'sales_lag_28', 'sales_lag_364',
    'sales_rolling_mean_7', 'sales_rolling_mean_14', 'sales_rolling_mean_28', 'sales_rolling_std_7',
    'promo_lag_1', 'promo_lag_2', 'promo_lag_7',
    'promo_lead_1', 'promo_lead_2',
    'promo_diff_1', 'promo_rolling_mean_7',
    'promo_intensity_7', 'promo_intensity_28'
]
print(f'特徴量数: {len(features)} 列')


## 4. 2シード・アベレージング学習 ＆ 再帰的推論 (Intel N100 最適化)

In [ ]:
families = train_df['family'].unique()
val_start_date = pd.to_datetime('2017-07-31')
val_end_date = pd.to_datetime('2017-08-15')
train_start_date = pd.to_datetime('2016-01-01')
test_start_date = pd.to_datetime('2017-08-16')
test_end_date = pd.to_datetime('2017-08-31')

# ドメイン補正: 元旦除外 ＆ 店舗52オープン前除外
is_new_year = (all_df['month'] == 1) & (all_df['day'] == 1)
is_pre_open_st52 = (all_df['store_nbr'] == 52) & (all_df['date'] < '2017-04-20')
trn_mask = (all_df['date'] >= train_start_date) & (all_df['date'] < val_start_date) & (~is_new_year) & (~is_pre_open_st52)

def run_recursive_inference(df_source, start_date, end_date, models_dict, is_val=True):
    pivot_sales = df_source.pivot(index='date', columns=['store_nbr', 'family'], values='log_sales').copy()
    dates = pd.date_range(start_date, end_date)
    step_predictions = []
    active_zero = safe_zero_pairs_val if is_val else safe_zero_pairs_tst

    for current_date in dates:
        day_mask = (df_source['date'] == current_date)
        day_df = df_source.loc[day_mask].copy()

        for lag in [1, 2, 3, 7, 14, 21, 28, 364]:
            target_d = current_date - timedelta(days=lag)
            alt_d = current_date - timedelta(days=28)
            lag_series = pivot_sales.loc[target_d] if target_d in pivot_sales.index else pivot_sales.loc[alt_d]
            day_df[f'sales_lag_{lag}'] = day_df.set_index(['store_nbr', 'family']).index.map(lag_series).values

        d_1 = current_date - timedelta(days=1)
        for window in [7, 14, 28]:
            d_win = current_date - timedelta(days=window)
            mean_series = pivot_sales.loc[d_win:d_1].mean(axis=0)
            day_df[f'sales_rolling_mean_{window}'] = day_df.set_index(['store_nbr', 'family']).index.map(mean_series).values

        d_win_7 = current_date - timedelta(days=7)
        std_series_7 = pivot_sales.loc[d_win_7:d_1].std(axis=0).fillna(0)
        day_df['sales_rolling_std_7'] = day_df.set_index(['store_nbr', 'family']).index.map(std_series_7).values

        day_preds = {}
        for fam in families:
            fam_mask = (day_df['family'] == fam)
            if not fam_mask.any(): continue
            X_day = day_df.loc[fam_mask, features].copy()
            X_lgb = X_day.copy()
            X_lgb['store_nbr'] = X_lgb['store_nbr'].astype('category')

            p_lgb = models_dict[fam]['lgb'].predict(X_day_lgb)
            p_cb = models_dict[fam]['cb'].predict(X_day)
            p_xgb = models_dict[fam]['xgb'].predict(X_day)
            p_sales = np.clip(np.expm1(0.50 * p_lgb + 0.30 * p_cb + 0.20 * p_xgb), 0, None)

            for idx, (st, val) in enumerate(zip(day_df.loc[fam_mask, 'store_nbr'], p_sales)):
                if (st, fam) in active_zero: p_sales[idx] = 0.0
            for idx_orig, p_val in zip(day_df.loc[fam_mask].index, p_sales):
                day_preds[idx_orig] = p_val

        day_df['pred_sales'] = pd.Series(day_preds)
        step_predictions.append(day_df[['id', 'date', 'store_nbr', 'family', 'sales', 'pred_sales']])

        for _, row in day_df.iterrows():
            pivot_sales.loc[current_date, (row['store_nbr'], row['family'])] = np.log1p(row['pred_sales'])

    return pd.concat(step_predictions, ignore_index=True)

# 2シード学習
seeds = [42, 2026]
val_preds = []
test_preds = []

for seed in seeds:
    print(f'Seed {seed} 学習中...')
    models = {}
    for fam in families:
        fam_trn_mask = trn_mask & (all_df['family'] == fam)
        fam_train_df = all_df.loc[fam_trn_mask].dropna(subset=features)
        X_trn = fam_train_df[features].copy()
        y_trn = fam_train_df['log_sales'].values

        lgb_m = lgb.LGBMRegressor(objective='regression', metric='rmse', learning_rate=0.05, num_leaves=31, feature_fraction=0.8, random_state=seed, verbose=-1, n_estimators=350, n_jobs=4)
        X_lgb = X_trn.copy()
        X_lgb['store_nbr'] = X_lgb['store_nbr'].astype('category')
        lgb_m.fit(X_lgb, y_trn)

        cb_m = cb.CatBoostRegressor(iterations=350, learning_rate=0.06, depth=6, loss_function='RMSE', task_type='CPU', thread_count=4, random_seed=seed, verbose=0)
        cb_m.fit(X_trn, y_trn, verbose=0)

        xgb_m = xgb.XGBRegressor(n_estimators=350, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, tree_method='hist', device='cpu', n_jobs=4, random_state=seed)
        xgb_m.fit(X_trn, y_trn, verbose=False)
        models[fam] = {'lgb': lgb_m, 'cb': cb_m, 'xgb': xgb_m}

    val_res = run_recursive_inference(all_df, val_start_date, val_end_date, models, is_val=True)
    val_preds.append(val_res['pred_sales'].values)
    test_res = run_recursive_inference(all_df, test_start_date, test_end_date, models, is_val=False)
    test_preds.append(test_res['pred_sales'].values)

final_val = np.mean(val_preds, axis=0)
final_test = np.mean(test_preds, axis=0)
val_rmsle = np.sqrt(mean_squared_error(np.log1p(val_res['sales']), np.log1p(final_val)))
print(f'2シード平均 手元検証 Val RMSLE: {val_rmsle:.5f}')


## 5. 提出ファイル作成

In [ ]:
sub = test_res[['id']].copy()
sub['sales'] = final_test
sub = sub.sort_values('id').reset_index(drop=True)
sub.to_csv('003_store-sales/submission_exp011.csv', index=False)
print('提出ファイル保存完了！')
print(sub.head(10))
